# 🚗 Entrenamiento YOLOv8s — Detección de Daños en Vehículos

**Dataset:** 13,787 imágenes | 22,808 anotaciones | 14 clases de daño

---

## ⚠️ ANTES DE EMPEZAR — Cambiar a GPU
1. Menú: `Entorno de ejecución → Cambiar tipo de entorno de ejecución`
2. Seleccionar **T4 GPU**
3. Guardar y reconectar

Luego ejecuta las celdas **en orden de arriba a abajo**.

In [ ]:
# ============================================================
# CELDA 1: Verificar GPU disponible
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU disponible:')
    for line in result.stdout.split('\n')[:15]:
        print(line)
else:
    print('❌ No se detectó GPU.')
    print('Ve a: Entorno de ejecución → Cambiar tipo → T4 GPU')
    raise RuntimeError('GPU requerida')

In [ ]:
# ============================================================
# CELDA 2: Instalar dependencias
# ============================================================
!pip install ultralytics pyyaml --quiet

import ultralytics, torch
print(f'✅ Ultralytics: {ultralytics.__version__}')
print(f'✅ PyTorch:     {torch.__version__}')
print(f'✅ CUDA:        {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# CELDA 3: Montar Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
print('\nContenido de My Drive (primeros 10 items):')
for item in os.listdir('/content/drive/MyDrive')[:10]:
    print(f'  {item}')

In [ ]:
# ============================================================
# CELDA 4: Configurar rutas
# ⚠️ SOLO CAMBIA BASE_DRIVE si subiste la carpeta a otro lugar
# ============================================================
import os

BASE_DRIVE = '/content/drive/MyDrive/proyecto_danos'
ZIP_PATH   = os.path.join(BASE_DRIVE, 'data', 'dataset_final.zip')
RUNS_DIR   = os.path.join(BASE_DRIVE, 'runs')

# DATASET_DIR y YAML_PATH se detectan automáticamente en Celda 5
DATASET_DIR = None
YAML_PATH   = None

if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / (1024**3)
    print(f'✅ ZIP encontrado: {ZIP_PATH}')
    print(f'   Tamaño: {size_gb:.2f} GB')
else:
    print(f'❌ ZIP no encontrado en: {ZIP_PATH}')
    print('   Sube dataset_final.zip a Drive → proyecto_danos → data → dataset_final.zip')
    raise FileNotFoundError(f'ZIP no encontrado: {ZIP_PATH}')

In [ ]:
# ============================================================
# CELDA 5: Extraer dataset + detectar estructura automáticamente
#
# NOTA TÉCNICA: Los ZIPs creados en Windows con Compress-Archive
# pueden tener rutas con backslashes (\) que el módulo zipfile de
# Python en Linux no interpreta como separadores de directorio.
# Por eso usamos el comando 'unzip' del sistema, que sí los maneja.
# ============================================================
import os, glob

EXTRACT_ROOT = '/content'

# Verificar si ya se extrajo en una ejecución anterior
yamls_existentes = glob.glob(f'{EXTRACT_ROOT}/**/data.yaml', recursive=True)
yamls_existentes = [y for y in yamls_existentes if 'drive' not in y]  # excluir Drive

if yamls_existentes:
    print('ℹ️  Dataset ya extraído. data.yaml encontrado en:')
    for y in yamls_existentes:
        print(f'   {y}')
else:
    print(f'📦 Extrayendo ZIP con unzip...')
    print(f'   Origen:  {ZIP_PATH}')
    print(f'   Destino: {EXTRACT_ROOT}')
    print('   (puede tardar 3-8 minutos)')

    # Intentar con unzip del sistema (maneja backslashes de Windows)
    ret = os.system(f'unzip -q "{ZIP_PATH}" -d {EXTRACT_ROOT}')

    if ret != 0:
        print('⚠️  unzip falló, intentando con Python zipfile + corrección de rutas...')
        import zipfile
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            for member in zf.infolist():
                # Convertir backslashes a forward slashes
                member.filename = member.filename.replace('\\', '/')
                zf.extract(member, EXTRACT_ROOT)
        print('✅ Extracción con zipfile completada')
    else:
        print('✅ Extracción con unzip completada')

# Buscar data.yaml en /content (excluyendo Drive)
yamls = [y for y in glob.glob(f'{EXTRACT_ROOT}/**/data.yaml', recursive=True)
         if 'drive' not in y]

if not yamls:
    print('\n❌ No se encontró data.yaml. Contenido de /content:')
    for item in sorted(os.listdir('/content')):
        item_path = os.path.join('/content', item)
        if os.path.isdir(item_path):
            sub = os.listdir(item_path)[:5]
            print(f'  📁 {item}/ → {sub}')
        else:
            print(f'  📄 {item}')
    raise FileNotFoundError('data.yaml no encontrado. Revisa la estructura del ZIP.')

# Usar el primer data.yaml encontrado
YAML_PATH   = yamls[0]
DATASET_DIR = os.path.dirname(YAML_PATH)

print(f'\n✅ data.yaml encontrado: {YAML_PATH}')
print(f'   DATASET_DIR: {DATASET_DIR}')

# Verificar estructura
print('\n📁 Estructura del dataset:')
total_imgs = 0
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    if os.path.exists(img_dir):
        exts = ('.jpg', '.jpeg', '.png', '.bmp')
        n_img = len([f for f in os.listdir(img_dir) if f.lower().endswith(exts)])
        n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.exists(lbl_dir) else 0
        print(f'   {split:6s}: {n_img:6d} imágenes | {n_lbl:6d} labels')
        total_imgs += n_img
    else:
        print(f'   {split:6s}: ⚠️ no encontrado ({img_dir})')

if total_imgs == 0:
    print('\n❌ Cero imágenes encontradas. Estructura dentro de DATASET_DIR:')
    for item in os.listdir(DATASET_DIR)[:20]:
        print(f'   {item}')
else:
    print(f'\n   TOTAL: {total_imgs} imágenes ✅')

In [ ]:
# ============================================================
# CELDA 6: Actualizar data.yaml con ruta correcta de Colab
# ============================================================
import yaml, os

if YAML_PATH is None:
    raise RuntimeError('Ejecuta la Celda 5 primero para detectar YAML_PATH')

print(f'Actualizando: {YAML_PATH}')

with open(YAML_PATH, 'r') as f:
    data_yaml = yaml.safe_load(f)

# Actualizar rutas para Colab (forward slashes, ruta absoluta)
data_yaml['path']  = DATASET_DIR
data_yaml['train'] = 'train/images'
data_yaml['val']   = 'valid/images'
data_yaml['test']  = 'test/images'

with open(YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print('\n✅ data.yaml actualizado:')
print(f"   path:  {data_yaml['path']}")
print(f"   train: {data_yaml['train']}")
print(f"   val:   {data_yaml['val']}")
print(f"   nc:    {data_yaml['nc']} clases")
print(f"   names: {data_yaml['names']}")

In [ ]:
# ============================================================
# CELDA 7: Entrenamiento YOLOv8s
# Tiempo estimado en T4 GPU: 4-6 horas para 100 épocas
# ============================================================
from ultralytics import YOLO
import os

if YAML_PATH is None:
    raise RuntimeError('Ejecuta las celdas 5 y 6 antes de entrenar')

os.makedirs(RUNS_DIR, exist_ok=True)

model = YOLO('yolov8s.pt')  # YOLOv8 Small: mejor balance velocidad/precisión

print('🚀 Iniciando entrenamiento...')
print(f'   Dataset YAML: {YAML_PATH}')
print(f'   Guardando en: {RUNS_DIR}')
print()

results = model.train(
    data=YAML_PATH,
    epochs=100,
    imgsz=640,
    batch=16,        # reducir a 8 si aparece error OOM (Out of Memory)
    patience=20,     # early stopping
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3,
    augment=True,
    flipud=0.0,      # NO voltear verticalmente
    fliplr=0.5,      # flip horizontal: válido para autos
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    name='danos_v1',
    project=RUNS_DIR,
    exist_ok=True,
    device=0,
    verbose=True,
    save=True,
    save_period=10,  # checkpoint cada 10 épocas
)

print('\n' + '='*50)
print('✅ ENTRENAMIENTO COMPLETADO')
print('='*50)
try:
    print(f"   mAP50:    {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.4f}")
    print(f"   mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
except:
    print('   (Ver métricas en Celda 8)')

In [ ]:
# ============================================================
# CELDA 8: Evaluar modelo en conjunto de validación
# ============================================================
from ultralytics import YOLO
import os

BEST_PT = os.path.join(RUNS_DIR, 'danos_v1', 'weights', 'best.pt')

if not os.path.exists(BEST_PT):
    print(f'❌ No se encontró best.pt en: {BEST_PT}')
else:
    model_eval = YOLO(BEST_PT)
    metrics = model_eval.val(data=YAML_PATH, split='val', verbose=False)

    print('📊 MÉTRICAS DE VALIDACIÓN:')
    print(f'   mAP50:     {metrics.box.map50:.4f}')
    print(f'   mAP50-95:  {metrics.box.map:.4f}')
    print(f'   Precisión: {metrics.box.mp:.4f}')
    print(f'   Recall:    {metrics.box.mr:.4f}')
    print()
    print('📋 AP50 por clase:')
    for i, clase in enumerate(metrics.names.values()):
        if i < len(metrics.box.ap50):
            ap = metrics.box.ap50[i]
            barra = '█' * int(ap * 20) + '░' * (20 - int(ap * 20))
            print(f'   {clase:20s} {barra} {ap:.3f}')

In [ ]:
# ============================================================
# CELDA 9: Visualizar curvas de entrenamiento
# ============================================================
from IPython.display import Image as IPImage, display
import os

TRAIN_DIR = os.path.join(RUNS_DIR, 'danos_v1')

for archivo, titulo in [
    ('results.png',                    'Curvas de entrenamiento'),
    ('confusion_matrix_normalized.png','Matriz de confusión normalizada'),
    ('PR_curve.png',                   'Curva Precisión-Recall'),
    ('F1_curve.png',                   'Curva F1'),
]:
    ruta = os.path.join(TRAIN_DIR, archivo)
    if os.path.exists(ruta):
        print(f'\n📈 {titulo}:')
        display(IPImage(ruta, width=800))
    else:
        print(f'⚠️  No encontrado: {archivo}')

In [ ]:
# ============================================================
# CELDA 10: Prueba rápida de inferencia
# Sube una foto de un auto dañado a Drive y ajusta la ruta
# ============================================================
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import os, glob

# ⚠️ Cambia esta ruta a tu imagen de prueba en Drive
IMAGEN_PRUEBA = '/content/drive/MyDrive/proyecto_danos/imagen_prueba.jpg'
BEST_PT       = os.path.join(RUNS_DIR, 'danos_v1', 'weights', 'best.pt')

if not os.path.exists(IMAGEN_PRUEBA):
    print(f'⚠️  Sube una imagen de prueba (auto dañado) a Drive:')
    print(f'   {IMAGEN_PRUEBA}')
elif not os.path.exists(BEST_PT):
    print('❌ best.pt no encontrado. Ejecuta entrenamiento primero.')
else:
    model_test = YOLO(BEST_PT)
    results = model_test.predict(
        IMAGEN_PRUEBA, conf=0.4, save=True,
        project='/content/predicciones', name='prueba', exist_ok=True,
    )
    print(f'🔍 Daños detectados: {len(results[0].boxes)}')
    for box in results[0].boxes:
        clase = model_test.names[int(box.cls[0])]
        print(f'   - {clase}: {float(box.conf[0]):.1%}')

    imgs = glob.glob('/content/predicciones/prueba/*.jpg') + \
           glob.glob('/content/predicciones/prueba/*.png')
    if imgs:
        print('\n🖼️ Resultado:')
        display(IPImage(imgs[0], width=700))

In [ ]:
# ============================================================
# CELDA 11: Descargar best.pt al equipo local
# ============================================================
import os
from google.colab import files

BEST_PT = os.path.join(RUNS_DIR, 'danos_v1', 'weights', 'best.pt')

if os.path.exists(BEST_PT):
    size_mb = os.path.getsize(BEST_PT) / (1024**2)
    print(f'✅ best.pt listo ({size_mb:.1f} MB)')
    print('⬇️  Iniciando descarga...')
    files.download(BEST_PT)
    print()
    print('📌 Próximo paso:')
    print('   Copia best.pt descargado a: p3Aplicaciones/models/best.pt')
    print('   Luego ejecuta: streamlit run app/main.py')
else:
    print(f'❌ No se encontró best.pt en: {BEST_PT}')
    print('   También puedes descargarlo directamente desde Google Drive.')